# 🏥 Machine Learning en Salud — Diabetes Pima Indian
## Taller Pre-Congreso CNIB 2025

---

> **¿Por qué usar ML para predecir diabetes?**
> La diabetes tipo 2 afecta a más de 500 millones de personas en el mundo. México tiene una de las prevalencias más altas: ~12% de la población adulta. El diagnóstico temprano es crítico para evitar complicaciones como ceguera, amputaciones e insuficiencia renal. Un modelo de ML puede ayudar a identificar pacientes en riesgo ANTES de que los síntomas sean evidentes.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📑 Dataset: Pima Indian Diabetes

**Origen:** Instituto Nacional de Diabetes y Enfermedades Digestivas y Renales (NIDDK), EE.UU.

**Población:** Mujeres de al menos 21 años de ascendencia indígena Pima (comunidad de Arizona/Sonora).

**¿Por qué los Pima?** Son uno de los grupos con mayor prevalencia de diabetes tipo 2 en el mundo (~50% de adultos). Esto se debe a la transición de su dieta tradicional baja en calorías a una dieta occidental, combinada con predisposición genética. Son un modelo de estudio clásico para la epidemiología de la diabetes.

**Variables del dataset:**
| Variable | Descripción médica |
|---|---|
| Pregnancies | Número de embarazos (factor de riesgo: diabetes gestacional) |
| Glucose | Concentración de glucosa en plasma a las 2h (prueba OGTT) |
| BloodPressure | Presión arterial diastólica (mm Hg) |
| SkinThickness | Grosor del pliegue cutáneo del tríceps (mm) — proxy de grasa corporal |
| Insulin | Insulina sérica a las 2h (μU/ml) |
| BMI | Índice de Masa Corporal (peso/talla²) |
| DiabetesPedigreeFunction | Puntuación de historia familiar de diabetes |
| Age | Edad en años |
| Outcome | **Variable objetivo**: 0=No diabetes, 1=Diabetes |


In [ ]:
import kagglehub
import os

# Descargamos el dataset de Kaggle
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
print("Ruta al dataset:", path)

dataset_path = "/kaggle/input/pima-indians-diabetes-database"
print(os.listdir(dataset_path))

In [ ]:
# Cargamos el CSV en un DataFrame de pandas
diabetes_df = pd.read_csv(os.path.join(dataset_path, "diabetes.csv"))
diabetes_df

---

## 📊 Análisis Exploratorio de Datos (EDA)

El EDA es el proceso de "conocer" los datos antes de modelar. Preguntas clave:
- ¿Cuántas muestras y variables tenemos?
- ¿Hay valores nulos o inconsistentes?
- ¿Cómo se distribuyen las variables?
- ¿Hay correlaciones entre variables?

> **Truco médico-estadístico 🧐:** En este dataset, los ceros en variables como Glucose, BloodPressure, Insulin, etc. son en realidad valores **faltantes** disfrazados — nadie puede tener 0 mm Hg de presión arterial y estar vivo. Esto es un ejemplo real de datos médicos sucios.


In [ ]:
# Nombres de las columnas
diabetes_df.columns

In [ ]:
# Tipo de dato y valores no nulos por columna
diabetes_df.info()

In [ ]:
# Estadísticas descriptivas
# Nota: observa los mínimos (min) en columnas como Glucose, BloodPressure —
# los valores 0 son físicamente imposibles → son NaN disfrazados
diabetes_df.describe()

In [ ]:
# Verificamos valores nulos explícitos (pandas-level)
# Los ceros-como-NaN no aparecen aquí, pero sí deberían tratarse
print("Valores nulos:\n", diabetes_df.isnull().sum())

In [ ]:
# Número de valores únicos por columna
# Útil para detectar variables con poca variabilidad
diabetes_df.nunique()

In [ ]:
# Distribución de cada variable con histograma + KDE
# KDE (Kernel Density Estimate): curva suavizada de la distribución

sns.set_style("darkgrid")

plt.figure(figsize=(14, len(diabetes_df.columns) * 3))
for idx, feature in enumerate(diabetes_df.columns, 1):
    plt.subplot(len(diabetes_df.columns), 2, idx)
    sns.histplot(diabetes_df[feature], kde=True)
    plt.title(f"Distribución: {feature}")

plt.tight_layout()
plt.show()

In [ ]:
# Pair Plot: visualización de relaciones entre TODAS las variables
# El color distingue Outcome=0 (no diabetes) de Outcome=1 (diabetes)
# Busca nubes de puntos separadas por color → eso indica buena separabilidad

plt.figure(figsize=(6, 6))
sns.pairplot(diabetes_df, hue="Outcome", palette="Set1", corner=True)
plt.show()

---

## ⚙️ Preparación de datos: separar features y target


In [ ]:
# Separamos la variable objetivo (lo que queremos predecir)
# de las variables independientes (los "síntomas" / mediciones)
diabetes_df_X = diabetes_df.drop("Outcome", axis=1)
diabetes_df_y = diabetes_df["Outcome"]

print("Features (X):", diabetes_df_X.shape)
print("Target (y):", diabetes_df_y.shape)
print("Proporción de clases:\n", diabetes_df_y.value_counts(normalize=True).round(3))

In [ ]:
# Mapa de correlación entre variables
# Valores cercanos a +1 → correlación positiva fuerte
# Valores cercanos a -1 → correlación negativa fuerte
# Valores cercanos a 0 → poca relación lineal

# Médicamente interesante: Glucose y Age tienen correlaciones más altas con el Outcome
plt.figure(figsize=(7, 6))
sns.heatmap(diabetes_df_X.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=2)
plt.title('Mapa de Correlación entre Variables')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_cols = diabetes_df_X.select_dtypes(include=[np.number]).columns

# Escalado estándar (media=0, std=1)
scaler_std = StandardScaler()
diabetes_df_X_scaled = diabetes_df_X.copy()
diabetes_df_X_scaled[num_cols] = scaler_std.fit_transform(diabetes_df_X[num_cols])

# Escalado MinMax (rango 0-1)
scaler_mm = MinMaxScaler()
diabetes_df_X_mm = diabetes_df_X.copy()
diabetes_df_X_mm[num_cols] = scaler_mm.fit_transform(diabetes_df_X[num_cols])

# Comparamos la distribución original vs escalada para cada variable
for column in num_cols:
    plt.figure(figsize=(3, 3))
    plt.scatter(diabetes_df[column],        diabetes_df["Outcome"], alpha=0.5, label="Original")
    plt.scatter(diabetes_df_X_scaled[column], diabetes_df["Outcome"], alpha=0.5, label="StandardScaler")
    plt.scatter(diabetes_df_X_mm[column],   diabetes_df["Outcome"], alpha=0.5, label="MinMaxScaler")
    plt.xlabel(column)
    plt.ylabel("Outcome")
    plt.legend(fontsize=6)
    plt.tight_layout()
    plt.show()

In [ ]:
# Histogramas después del escalado estándar
# Las distribuciones deberían estar centradas alrededor de 0

sns.set_style("darkgrid")
plt.figure(figsize=(14, len(diabetes_df_X_scaled.columns) * 3))
for idx, feature in enumerate(diabetes_df_X_scaled.columns, 1):
    plt.subplot(len(diabetes_df_X_scaled.columns), 2, idx)
    sns.histplot(diabetes_df_X_scaled[feature], kde=True)
    plt.title(f"{feature} (escalado)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# División 64% train / 16% val / 20% test con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    diabetes_df_X_scaled, diabetes_df_y,
    test_size=0.2, stratify=diabetes_df_y, random_state=1
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2, stratify=y_train, random_state=1
)

labels = ['Train', 'Validation', 'Test']
sizes  = [len(y_train), len(y_val), len(y_test)]
plt.figure(figsize=(5,5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=['#ff9999', '#66b3ff', '#99ff99'])
plt.title('División del Dataset — Diabetes')
plt.show()

---

## 🤖 Definición de Modelos

Entrenamos 8 modelos clásicos de ML para comparar su desempeño. Aquí un resumen rápido de cada uno:

| Modelo | Idea principal | Ventaja médica |
|---|---|---|
| **Logistic Regression** | Línea (o hiperplano) de separación lineal con probabilidades | Interpretable: los coeficientes tienen significado clínico |
| **KNN** | Clasifica según los k vecinos más cercanos | Simple, no asume distribución |
| **SVM** | Maximiza el margen entre clases | Robusto en espacios de alta dimensión |
| **Naive Bayes** | Probabilidad bayesiana asumiendo independencia | Funciona bien con pocos datos |
| **Decision Tree** | Árbol de reglas tipo "si/entonces" | Muy interpretable por médicos |
| **Random Forest** | Ensemble de árboles de decisión | Robusto, maneja bien el overfitting |
| **XGBoost** | Gradient boosting extremo | Alto rendimiento, usado en competencias |
| **Perceptron** | Red neuronal de una capa | Base histórica de las redes profundas |


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import Perceptron
from sklearn import metrics

# Creamos instancias de cada modelo con configuración por defecto
logreg_model      = LogisticRegression()
knn_model         = KNeighborsClassifier()
svm_model         = SVC()
nb_model          = GaussianNB()
dt_model          = DecisionTreeClassifier()
rf_model          = RandomForestClassifier()
xgb_model         = XGBClassifier()
perceptron_model  = Perceptron()

ML_models = [logreg_model, knn_model, svm_model, nb_model,
             dt_model, rf_model, xgb_model, perceptron_model]

---

## 🏋️ Entrenamiento

El método `.fit(X_train, y_train)` le muestra al modelo los pares (características, etiqueta) para que aprenda los patrones. Cada modelo lo hace de manera diferente (ecuaciones lineales, árboles, etc.) pero la interfaz en sklearn es uniforme.


In [ ]:
# Entrenamos todos los modelos con los datos de entrenamiento
# Esto puede tardar unos segundos

for model in ML_models:
    model.fit(X_train, y_train)
    print(f"✓ {model.__class__.__name__} entrenado")

---

## 📏 Evaluación y Predicciones

**Accuracy** = (Predicciones correctas) / (Total de predicciones). Es la métrica más intuitiva pero puede ser engañosa con datos desbalanceados.

> **Ejemplo médico:** Si el 90% de los pacientes no tiene diabetes, un modelo que siempre dice "no diabetes" tendría 90% de accuracy ¡sin aprender nada! Por eso también usamos precision, recall y F1.


In [ ]:
# Accuracy en train, validación y test para cada modelo
# train >> test → overfitting (sobreajuste al conjunto de entrenamiento)
# train ≈ test → buen ajuste

for model in ML_models:
    print(f"{model.__class__.__name__}")
    print(f"  Train: {model.score(X_train, y_train):.3f}")
    print(f"  Val:   {model.score(X_val, y_val):.3f}")
    print(f"  Test:  {model.score(X_test, y_test):.3f}")
    print()

In [ ]:
# Matrices de confusión para cada modelo en el conjunto de prueba
# La matriz muestra:
#   Verdaderos Positivos (TP): predicho Sí, real Sí
#   Verdaderos Negativos (TN): predicho No, real No
#   Falsos Positivos (FP): predicho Sí, real No  ← error tipo I
#   Falsos Negativos (FN): predicho No, real Sí  ← error tipo II (el más peligroso en medicina)

for model in ML_models:
    y_pred = model.predict(X_test)
    confusion_matrix = metrics.confusion_matrix(y_test, y_pred)
    cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=confusion_matrix,
                                                 display_labels=[False, True])
    cm_display.plot(cmap='Blues', values_format='d')
    plt.title(model.__class__.__name__)
    plt.show()

In [ ]:
# Ordenamos los modelos por accuracy en test (de mayor a menor)
ordered_models = sorted(ML_models,
                         key=lambda model: model.score(X_test, y_test),
                         reverse=True)

print("Ranking de modelos por accuracy en test:")
for i, model in enumerate(ordered_models, 1):
    print(f"  {i}. {model.__class__.__name__}: {model.score(X_test, y_test):.3f}")

---

## 🔎 Selección de Características

Probamos si reducir a las 3 variables más relevantes mejora o empeora los modelos.

> **Reflexión médica 💊:** En clínica, un médico no necesita 8 variables para diagnosticar diabetes. La glucosa en ayunas y el IMC ya dan mucha información. Este experimento cuantifica cuánto vale cada variable.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Seleccionamos las 3 características más discriminativas
selector = SelectKBest(f_classif, k=3)
selector.fit(diabetes_df_X_scaled, diabetes_df_y)

scores = selector.scores_
plt.figure(figsize=(10,4))
sns.barplot(x=diabetes_df_X_scaled.columns, y=scores, palette='Blues_d',
            order=diabetes_df_X_scaled.columns[np.argsort(scores)])
plt.xlabel('Características', fontsize=14)
plt.ylabel('Score F', fontsize=14)
plt.xticks(rotation=90)
plt.title("Relevancia de características (ANOVA F-score)")
plt.tight_layout()
plt.show()

In [ ]:
# Reducimos el dataset a las 3 mejores características
diabetes_df_X_top3 = diabetes_df_X_scaled.iloc[:, selector.get_support(indices=True)]
print("Forma original:", diabetes_df_X_scaled.shape)
print("Después de selección:", diabetes_df_X_top3.shape)
print("Columnas elegidas:", diabetes_df_X_top3.columns.tolist())

In [ ]:
# Nueva división del dataset reducido
X_train_top3, X_test_top3, y_train_top3, y_test_top3 = train_test_split(
    diabetes_df_X_top3, diabetes_df_y,
    test_size=0.2, stratify=diabetes_df_y, random_state=1)
X_train_top3, X_val_top3, y_train_top3, y_val_top3 = train_test_split(
    X_train_top3, y_train_top3, test_size=0.2, stratify=y_train_top3, random_state=1)

print('X_train:', X_train_top3.shape, 'X_val:', X_val_top3.shape, 'X_test:', X_test_top3.shape)

In [ ]:
# Re-entrenamos todos los modelos con solo las 3 mejores características
for model in ML_models:
    model.fit(X_train_top3, y_train_top3)
    print(f"{model.__class__.__name__}")
    print(f"  Train: {model.score(X_train_top3, y_train_top3):.3f} | "
          f"Val: {model.score(X_val_top3, y_val_top3):.3f} | "
          f"Test: {model.score(X_test_top3, y_test_top3):.3f}")
    print()

In [ ]:
ordered_models_top3 = sorted(ML_models,
    key=lambda m: m.score(X_test_top3, y_test_top3), reverse=True)
print("Ranking con TOP-3 features:")
for i, m in enumerate(ordered_models_top3, 1):
    print(f"  {i}. {m.__class__.__name__}: {m.score(X_test_top3, y_test_top3):.3f}")

---

## 📉 Extracción de Características con PCA


In [ ]:
from sklearn.decomposition import PCA

# Reducimos las 3 características a 2 componentes principales
pca = PCA(n_components=2)
diabetes_pca = pca.fit_transform(diabetes_df_X_top3)

plt.figure(figsize=(8, 4))
plt.scatter(diabetes_pca[:, 0], diabetes_pca[:, 1],
            c=diabetes_df_y, cmap='viridis')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('PCA del Dataset de Diabetes')
plt.colorbar(label='Outcome (0=No diabetes, 1=Diabetes)')
plt.show()

In [ ]:
# División y re-entrenamiento con datos reducidos por PCA
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    diabetes_pca, diabetes_df_y, test_size=0.2, stratify=diabetes_df_y, random_state=1)
X_train_pca, X_val_pca, y_train_pca, y_val_pca = train_test_split(
    X_train_pca, y_train_pca, test_size=0.2, stratify=y_train_pca, random_state=1)

for model in ML_models:
    model.fit(X_train_pca, y_train_pca)
    print(f"{model.__class__.__name__}: "
          f"Train={model.score(X_train_pca, y_train_pca):.3f} | "
          f"Val={model.score(X_val_pca, y_val_pca):.3f} | "
          f"Test={model.score(X_test_pca, y_test_pca):.3f}")

In [ ]:
ordered_models_pca = sorted(ML_models,
    key=lambda m: m.score(X_test_pca, y_test_pca), reverse=True)
print("Ranking con PCA:")
for i, m in enumerate(ordered_models_pca, 1):
    print(f"  {i}. {m.__class__.__name__}: {m.score(X_test_pca, y_test_pca):.3f}")

---

## 🏆 Tabla de Puntos por Experimento

Acumulamos puntos según la posición en cada ranking (más puntos = mejor modelo consistente).


In [ ]:
# Sistema de puntos: 8 puntos al 1er lugar, 7 al 2do, etc.
Models_points = {m.__class__.__name__: 0 for m in ML_models}

for idx in range(len(ML_models)):
    Models_points[ordered_models[idx].__class__.__name__]     += len(ML_models) - idx
    Models_points[ordered_models_top3[idx].__class__.__name__] += len(ML_models) - idx
    Models_points[ordered_models_pca[idx].__class__.__name__]  += len(ML_models) - idx

Models_points = dict(sorted(Models_points.items(), key=lambda x: x[1], reverse=True))
for name, pts in Models_points.items():
    print(f"  {name}: {pts} puntos")

---

## 🧠 Red Neuronal con Keras (MLP)

**Perceptrón Multicapa (MLP):** Una red neuronal feedforward con capas ocultas. Inspirada vagamente en la estructura del cerebro, es capaz de aprender representaciones no lineales complejas.

**Arquitectura usada:**
```
Input (3) → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Sigmoid)
```

- **ReLU** (Rectified Linear Unit): `max(0, x)` — activa neuronas positivas, apaga las negativas. Rápida y evita el problema del gradiente que desaparece.
- **Sigmoid**: aplasta la salida a [0,1] — perfecta para clasificación binaria (diabetes: sí/no).
- **Binary crossentropy**: función de pérdida para problemas de clasificación binaria.
- **Early Stopping**: detiene el entrenamiento cuando el val_loss deja de mejorar — evita overfitting automáticamente.


In [ ]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import ModelCheckpoint, EarlyStopping
import keras

# Hiperparámetros del modelo
epochs        = 50
batch_size    = 16
learning_rate = 0.01

In [ ]:
# Definición de la arquitectura de la red neuronal
# input_shape debe coincidir con el número de características de entrada
model = Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train_top3.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')   # Salida: probabilidad de diabetes
])

print(model.summary())

In [ ]:
# Compilación: definimos el optimizador, la función de pérdida y la métrica
# Adam: variante eficiente del descenso de gradiente estocástico
opt = keras.optimizers.Adam(learning_rate=learning_rate)
model.compile(
    loss='binary_crossentropy',  # Para clasificación binaria
    optimizer=opt,
    metrics=['accuracy']
)

In [ ]:
# Early Stopping: si val_loss no mejora por 20 épocas, detenemos el entrenamiento
# y restauramos los mejores pesos encontrados hasta ese momento
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    mode='min',
    restore_best_weights=True
)

In [ ]:
# Checkpoint: guarda el modelo cada vez que encuentra un mejor val_accuracy
checkpoint = ModelCheckpoint(
    'diabetes-model.best.h5',
    monitor='val_acc',
    verbose=1,
    save_best_only=True,
    mode='max'
)

### Entrenamiento de la Red Neuronal

In [ ]:
history = model.fit(
    X_train_top3, y_train_top3,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_top3, y_val_top3),
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
# Graficamos las curvas de aprendizaje
# Si train_accuracy sube pero val_accuracy se estanca o baja → overfitting

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy',      color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title('Accuracy durante el entrenamiento')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss',      color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Loss durante el entrenamiento')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.suptitle("Curvas de Aprendizaje — Red Neuronal", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación final en el conjunto de prueba (datos que el modelo NUNCA ha visto)
scores = model.evaluate(X_test_top3, y_test_top3)
print(f"Test Loss:     {scores[0]:.3f}")
print(f"Test Accuracy: {scores[1]:.3f}")